# Phase 6.5 — Task 6.5.5 controls and the per-arm curve deliverable

The registered seven-arm run (`phase6_5_sciplex3_colab.ipynb`) left three things unmeasured. This notebook measures them, on the **same** split, frame, grids, bridges and seeds — it calls `run_phase6_5.build_context`, the identical pipeline prefix, so nothing is re-tuned and no registered number moves.

**1. The negative control (Task 6.5.5).** Pseudo-treatment is assigned at random among the 24 vehicle evaluation wells, on distinct plates, three per draw, mirroring the real contrast's imbalance. Because assignment is independent of the wells, the true contrast is **exactly the zero function**. This is the only target in the whole phase that needs no operational definition and no oracle. The plan is unambiguous about its status: *"Report this before any treated contrast. A failure here halts the phase."*

**2. The null-thinning control (Task 6.5.5).** At `p = 1.0` the thinned data *is* the full data, so a correct correction must be the identity. The registered run could only see this indirectly through coverage. Here we measure it directly: the sup-norm distance between each corrected arm's estimate and the blind estimate, on the same replicate and the same units. The registered falsification condition is "the correction is non-trivial at `p = 1.0`", and this is the number that condition is about.

**3. The estimate curves.** `evaluate._cov_row` keeps only three scalars and discards `band.estimate`, so the blind / corrected / DTM-augmented conclusion comparison required as a deliverable by `docs/phase6_analysis.md` item 4 cannot be produced from the existing artifacts, and neither can any plot. This run persists every arm's ψ curve, plus the per-unit evaluation curves so future re-analysis costs seconds instead of a full bridge re-fit.

Runtime is dominated by the shared prefix (all 4,614 calibration units × 4 retention levels × Alpha and DTM curves), the same several hours as the registered run. The controls themselves are cheap once the bridges exist.

**Not covered here:** the positive control and the dose-monotonicity check, the other two bullets of Task 6.5.5. Both need contrasts outside the registered evaluation split (other compounds, and sci-Plex 2's seven-dose ladder), which is a separate data build. They remain open in `docs/phase6_5_analysis.md`.

In [ ]:
# Install only the packages needed by the applied runner.
%pip -q install anndata h5py gudhi joblib

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Option A: clone the repository (needs the commit that adds
# btate/applied/controls.py and btate/applied/run_controls.py).
# Option B: set REPO_DIR to a Drive copy if that commit is not yet pushed.
REPO_DIR = Path('/content/btate')
if not (REPO_DIR / 'btate/applied/run_controls.py').exists():
    subprocess.run(['git', 'clone', 'https://github.com/hugogobato/btate.git',
                    str(REPO_DIR)], check=True)
assert (REPO_DIR / 'btate/applied/run_controls.py').exists(), (
    'Upload or clone the repository containing btate/applied/run_controls.py'
)

OUT = Path('/content/phase6_5_controls_results')
print('repo:', REPO_DIR)

In [ ]:
# Resolve the registered sci-Plex 3 file: prefer a Drive copy if present,
# otherwise stream the registered scPerturb artifact from Zenodo to disk.
# Same h5ad as the registered run (docs/phase6_5_registration.md).
import urllib.request

DRIVE_H5AD = Path('/content/drive/MyDrive/srivatsan_2020_sciplex3.h5ad')
H5AD = Path('/content/srivatsan_2020_sciplex3.h5ad')
if DRIVE_H5AD.exists():
    H5AD = DRIVE_H5AD
    print('Using registered data from Drive:', H5AD)
if not H5AD.exists():
    url = ('https://zenodo.org/records/13350497/files/'
           'SrivatsanTrapnell2020_sciplex3.h5ad?download=1')
    print('Downloading registered data (2.4 GB, streams to disk)...')
    urllib.request.urlretrieve(url, H5AD)
    assert H5AD.stat().st_size > 2**30, 'Downloaded file looks truncated'
print('data:', H5AD, H5AD.stat().st_size / 2**30, 'GiB')

In [ ]:
# Conservative resource guard. Do not raise JOBS on a shared runtime.
# BLAS pinning is not optional here: the Godambe sandwich is near-singular
# and a thread-count difference moves bands by percent-level amounts.
import psutil
available = psutil.virtual_memory().available
minimum = 5 * 2**30
if available < minimum:
    raise RuntimeError(f'Only {available / 2**30:.2f} GiB available; refusing to risk OOM')

os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

JOBS = 2 if available >= 10 * 2**30 else 1
FRAME_CAP = 50_000
N_REPLICATES = 100
N_REP = 20
N_PSEUDO_TREATED = 3   # mirrors the 3 real Alisertib wells
print(f'available RAM: {available / 2**30:.2f} GiB; jobs={JOBS}')

In [ ]:
# Optional smoke test (~minutes, not interpretable): caps the calibration
# pool so the whole path is exercised before committing to the full run.
# Set SMOKE = False for the real run.
SMOKE = True
if SMOKE:
    cmd = [
        sys.executable, '-u', '-m', 'btate.applied.run_controls',
        '--h5', str(H5AD), '--out', str(OUT) + '_smoke', '--jobs', str(JOBS),
        '--frame-cap', '5000', '--n-replicates', '4', '--n-rep', '12',
        '--max-calib', '40',
    ]
    subprocess.run(cmd, cwd=REPO_DIR, check=True)
    print('\nSmoke run finished. Set SMOKE = False and re-run the next cell.')

In [ ]:
# Full registered control run. Several hours: the cost is the shared prefix
# (all 4,614 calibration units get every retention level's Alpha/DTM curves),
# not the controls.
cmd = [
    sys.executable, '-u', '-m', 'btate.applied.run_controls',
    '--h5', str(H5AD), '--out', str(OUT), '--jobs', str(JOBS),
    '--frame-cap', str(FRAME_CAP), '--n-replicates', str(N_REPLICATES),
    '--n-rep', str(N_REP), '--n-pseudo-treated', str(N_PSEUDO_TREATED),
]
subprocess.run(cmd, cwd=REPO_DIR, check=True)

In [ ]:
# Mechanical completeness checks before reading any scientific conclusion.
import json
import pandas as pd

art = json.loads((OUT / 'controls_artifacts.json').read_text())
assert art['n_eval'] == 27
assert art['n_control_eval'] == 24
assert art['n_pseudo_treated'] == N_PSEUDO_TREATED

null_rows = pd.read_csv(OUT / 'negative_control_rows.csv')
est_rows = pd.read_csv(OUT / 'estimate_rows.csv')
assert not bool(null_rows['failed'].any()), 'arms failed in the negative control'
print('negative-control rows:', len(null_rows), ' estimate rows:', len(est_rows))
print(art)

## Gate 1 — negative control (halts the phase on failure)

The true contrast is exactly zero. Each arm's simultaneous band should contain the zero function at its nominal 95% rate. Read `cov_zero_rate` against 0.95, **not** against the M1 reference level: unlike `cov_sim_full`, this target is exact, so nominal is the right yardstick.

Two distinct failure modes to keep apart. **Under-coverage** means the band is too narrow or the estimate is biased away from zero, and it invalidates that arm everywhere. **Over-coverage at 1.00 with a wide band** means the arm is not falsifiable on this control — it would cover zero whatever the data said — which is not a pass so much as an absence of evidence. `max_abs_estimate_mean` separates them: it is how far the arm's point estimate wanders from zero when nothing is happening.

In [ ]:
null_agg = pd.read_csv(OUT / 'negative_control_aggregate.csv')
print(null_agg.to_string(index=False))

PRIMARY_P = 0.25
print('\n--- Gate 1 verdict at the primary p = 0.25 ---')
cell = null_agg[null_agg['p'] == PRIMARY_P]
for _, r in cell.iterrows():
    covers = r['cov_zero_cp_lower'] <= 0.95 <= r['cov_zero_cp_upper']
    verdict = 'PASS' if covers else ('UNDER-COVERS' if r['cov_zero_rate'] < 0.95
                                     else 'OVER-COVERS')
    print(f"{r['method']:28s} cov_zero={r['cov_zero_rate']:.2f} "
          f"[{r['cov_zero_cp_lower']:.2f}, {r['cov_zero_cp_upper']:.2f}] "
          f"width={r['band_width_mean']:.4f} "
          f"|est|max={r['max_abs_estimate_mean']:.4f}  {verdict}")

under = cell[cell['cov_zero_cp_upper'] < 0.95]['method'].tolist()
if under:
    print('\nHALT CONDITION MET for:', under)
    print('Task 6.5.5: "A failure here halts the phase." These arms are broken')
    print('independently of any bridge claim and cannot be reported as corrections.')
else:
    print('\nNo arm under-covers zero at p = 0.25; the halt condition is not met.')

## Gate 2 — null-thinning control (registered falsification condition)

At `p = 1.0` the thinned counts are the full counts. A correction that is the identity there leaves the estimate where blind AIPW put it, so `sup_shift_vs_blind` ≈ 0. Anything materially larger means the bridge is "correcting" data it was told is uncorrupted, which is the registered falsification condition in `docs/phase6_5_registration.md` §11.

Scale the shift against something meaningful rather than eyeballing it: the printout compares it to the peak of `psi_full` (0.0224 in the registered run) and to the oracle band width (0.0064). A shift comparable to the effect itself is not a rounding error.

In [ ]:
import numpy as np

thin = pd.read_csv(OUT / 'null_thinning_aggregate.csv')
psi_full = np.load(OUT / 'psi_full.npy')
peak = float(np.max(np.abs(psi_full)))
oracle_width = float(pd.read_csv(OUT / 'estimate_rows.csv')
                     .query("method == 'M1_oracle_full_aipw'")['band_width'].mean())
print(f'|psi_full|_max = {peak:.4f}   mean oracle band width = {oracle_width:.4f}\n')

at_one = thin[thin['p'] == 1.0]
print('--- Gate 2 verdict at p = 1.0 (correction must be the identity) ---')
for _, r in at_one.iterrows():
    if r['method'] in ('M1_oracle_full_aipw', 'M2_blind_aipw'):
        continue
    shift = r['sup_shift_vs_blind_mean']
    flag = 'NON-TRIVIAL' if shift > 0.1 * peak else 'ok'
    print(f"{r['method']:28s} sup_shift_vs_blind={shift:.5f} "
          f"({shift / peak:5.1%} of |psi_full|_max, "
          f"{shift / oracle_width:4.1f}x oracle width)  {flag}")

print('\nFull ladder (a correction should grow as p falls, and vanish at p = 1):')
print(thin.pivot_table(index='method', columns='p',
                       values='sup_shift_vs_blind_mean').to_string())

## Deliverable — blind vs corrected vs DTM-augmented conclusions

`docs/phase6_analysis.md` item 4 requires this comparison as a deliverable, not an option. With the estimate curves persisted it is finally computable: for each arm, how far its ψ lands from the full-depth reference, where it puts the peak, and how large it says the effect is.

In [ ]:
print('--- Conclusion comparison at the primary p = 0.25 ---')
cmp = (est_rows[est_rows['p'] == PRIMARY_P]
       .groupby('method')
       .agg(cov_sim_full=('cov_sim_full', 'mean'),
            interval_score=('interval_score', 'mean'),
            band_width=('band_width', 'mean'),
            sup_err_vs_psi_full=('sup_err_vs_psi_full', 'mean'),
            peak_abs_estimate=('peak_abs_estimate', 'mean'),
            argmax_t=('argmax_t', 'median'))
       .sort_values('interval_score'))
print(cmp.to_string())
print(f'\nreference: |psi_full|_max = {peak:.4f} at t = '
      f"{np.asarray(np.load(OUT / 'estimate_curves.npz')['grid'])[int(np.argmax(np.abs(psi_full)))]:.4f}")
print('\nRank by interval score, never by coverage alone: the corrected arms')
print('over-cover by widening, and the interval score is what prices that.')

In [ ]:
# Mean estimated curve per arm against the full-depth reference.
import matplotlib.pyplot as plt

npz = np.load(OUT / 'estimate_curves.npz')
curves, grid = npz['curves'], npz['grid']
sel = est_rows['p'] == PRIMARY_P
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(grid, psi_full, 'k-', lw=2.5, label='psi_full (reference)', zorder=5)
for method in ['M2_blind_aipw', 'M4_bridge_bayes', 'M4_dtm_bridge_bayes',
               'M5_bridge_freq_mi', 'M5_dtm_bridge_freq_mi']:
    rows_m = np.flatnonzero(sel & (est_rows['method'] == method))
    if rows_m.size:
        ax.plot(grid, curves[rows_m].mean(axis=0), lw=1.4, alpha=0.85,
                label=method)
ax.set_xlabel('filtration t'); ax.set_ylabel('psi(t)')
ax.set_title(f'Mean estimated TATE per arm at p = {PRIMARY_P}')
ax.legend(fontsize=8); ax.axhline(0, color='0.7', lw=0.6)
fig.tight_layout(); fig.savefig(OUT / 'psi_by_arm.png', dpi=150)
plt.show()

In [ ]:
# Bundle all artifacts and download automatically when running in Colab.
import shutil
archive = shutil.make_archive('/content/phase6_5_controls_artifacts', 'zip',
                              root_dir=OUT)
output_file = archive
try:
    from google.colab import files
    files.download(output_file)
    print("Downloaded:", output_file)
except Exception as e:
    print("(Not on Colab / download skipped):", e)
print('Artifacts:', OUT)